# **Who**: David

## **Task**: find out what phrases/keywords are important for the positively annotated class with respect to the negatives.

Ingredients:
* simply word frequencies: first remove stopwords and extract the word counts (use sklearn's CountVectorizer perhaps), maybe do this per section of the text
* use Jacopo's models in combination with a tool like LIME or SHAP.
* aggregate the embeddings of the medical terms in the texts 

In [ ]:
import pandas as pd

import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer

import matplotlib.pyplot as plt

from scipy import stats

In [ ]:
# Necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [4]:
df_consults = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822\consult_20240822.sas7bdat',
        encoding='latin1')

df_consults = df_consults.drop(['Consult_id', 'ConsultStatus', 'type_code_original',
    'type_display_original','created', 'title', 'author_usercode',
    'initial_author_usercode', 'custodian_Organization_value',
    'SpecialismeNaam', 'initial_custodian_Organization_v',
    'SpecialismeNaamInitial', 'LocationCode', 'LocationName',
    'PatientProtected', 'section_title_code_original',
    'section_title_display_original', 'section_text',
    'section_text_contentType', 'orderedBy', 'index_date'],
    axis='columns')

df_consults = df_consults.groupby('studyId_0831').last()

df_consults_text = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

df_merged = pd.merge(df_consults_text, df_consults, right_index=True,left_on='studyId_0831',how='left')

In [5]:
df_merged['delta'] = (df_merged.date_x - df_merged.date_y).dt.days

df_merged_filt = df_merged.query('delta >= -365 and delta <= 0')

In [6]:
# Funzione per ottenere il testo più lungo e il rispettivo delta
def longest_text(group):
    longest = group.loc[group['text'].str.len().idxmax()]
    return pd.Series({
        'text': longest['text'],
        'delta': longest['delta']
    })

In [7]:
# Raggruppa per patient_id e applica la funzione
df_merged_f_g = df_merged_filt.groupby('studyId_0831').apply(longest_text, include_groups=False)

In [8]:
df_afsprak = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805\afspraken_20240805.sas7bdat',
        encoding='latin1')

df_afsprak.studyId_0831 = df_afsprak.studyId_0831.astype('uint64')

df_afsprak = df_afsprak.filter(items=['studyId_0831', 'index_AfspCode'])

df_afsprak = df_afsprak.groupby('studyId_0831', sort=True).agg(lambda x: x.to_list())

In [9]:
def isEligible(x:pd.Series):
    if 'NFALG' in x or 'NFGEH' in x or 'NFVMI+' in x or 'NFVAL+' in x:
        return 1
    else:
        return 0

In [ ]:
df_afsprak['label'] = df_afsprak.index_AfspCode.apply(isEligible)

df_afsprak = df_afsprak[['label']]

df_afsprak

In [11]:
df_afsprak = df_afsprak.groupby('studyId_0831').last()

In [12]:
df_cons_afs = pd.merge(df_merged_f_g, df_afsprak, 
    right_index=True, left_on='studyId_0831', how='left')

In [ ]:
plt.figure()
plt.show(df_cons_afs['delta'].hist(bins=50))

In [ ]:
df_cons_afs['text'].str.split().str.len().sort_values(ascending=False).hist()

### Stopwords and numbers removal from the consults dataset

In [ ]:
# Rimuovi tutti i numeri dal testo
df_cons_afs['text'] = df_cons_afs['text'].str.replace(r'\d+', '', regex=True)

# Carica le stopwords per la lingua olandese
stop_words = stopwords.words('dutch')

In [ ]:
# Funzione per rimuovere stopwords e punteggiatura
def remove_stopwords(text):
    # Rimuovere punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Rimuovere stopwords e convertire in minuscolo
    text = ' '.join([word for word in text.lower().split() if word not in stop_words])
    
    return text

In [ ]:
# Applica la funzione al testo
df_cons_afs['clean_text'] = df_cons_afs['text'].apply(remove_stopwords)

# Contare le parole in ciascun testo
df_cons_afs['word_count'] = df_cons_afs['clean_text'].str.split().str.len()

# Ordinare il DataFrame in base al conteggio delle parole
df_cons_afs = df_cons_afs.sort_values(by='word_count')

In [ ]:
result = df_cons_afs['clean_text'].str.split().str.len().sort_values(ascending=False)

print(result.to_string(index=True))

In [ ]:
Id = 43804529669

'''
print(df_merged_f_g.loc[Id, 'text'].split())
print('')
print(df_merged_f_g.loc[Id, 'clean_text'].split())
print('')
print(df_merged_f_g.loc[Id, 'clean_text'])
print('')
'''
print(df_cons_afs.loc[Id, 'clean_text'])

### Word Frequency Analysis

Devo fare riferimento a https://docs.scipy.org/doc/scipy/tutorial/sparse.html

Devo cercare il miglior metodo per comparare count distribution

In [19]:
# Crea un'istanza di CountVectorizer
vectorizer = CountVectorizer(ngram_range=(1, 3), min_df=0.01, max_df=0.99)

# Adatta e trasforma i testi delle consultazioni
word_counts = vectorizer.fit_transform(df_cons_afs['clean_text'])

Sparse matrix elaboration

In [ ]:
# Calcolo del numero totale di elementi
total_elements = word_counts.shape[0] * word_counts.shape[1]

print("Number of zeroes:", total_elements - word_counts.count_nonzero())
print("Values different from zero:", word_counts.count_nonzero())
#print(word_counts.nnz - word_counts.count_nonzero())

In [21]:
words_occ = word_counts.sum(axis=0)

# Ottieni i nomi degli n-gram
ngram_names = vectorizer.get_feature_names_out()

# Crea un DataFrame per combinare i n-gram e i loro conteggi
ngram_counts = pd.DataFrame(words_occ.T, index=ngram_names, columns=['count'])

ngram_counts = ngram_counts.sort_values(by='count', ascending=False)

In [22]:
# Funzione per contare gli n-grammi
def count_ngrams(text, ngram_list):
    counts = {ngram: text.count(ngram) for ngram in ngram_list}
    return pd.Series(counts)

In [23]:
# Estrai la lista degli n-gram
lista_ngram = ngram_counts.index.tolist()

# Applicare la funzione al dataframe
ngram_counts = df_cons_afs['clean_text'].apply(lambda x: count_ngrams(x, lista_ngram))

# Unire i risultati al dataframe originale
result_df = pd.concat([df_cons_afs, ngram_counts], axis=1)

### Statistical Analysis

In [24]:
statistics = []

for ngram in lista_ngram:
    column_0 = result_df.loc[result_df['label'] == 0, ngram]
    column_1 = result_df.loc[result_df['label'] == 1, ngram]
    
    stat, p_value = stats.ks_2samp(column_0, column_1)
    
    statistics.append({'p-value': p_value, 'D-statistic': stat})

df_stats = pd.DataFrame(statistics, index=lista_ngram)

In [ ]:
df_stats.sort_values(by='D-statistic', ascending=False)[:50]

In [ ]:
significative_words = df_stats[df_stats['p-value'] < 0.01/4000].index.tolist()
print(significative_words[:50])

print(len(significative_words))